In [ ]:


Got it. Let me write every line of `pre-train.py` in the chat, interleaved with deep explanations and visualizations. Nothing gets written to file until we've reasoned through everything together.

---

## Section 1: Script Header + Imports

```python
"""
NanoSeek Stage 1 Pre-Training Script.

Usage:
    Single GPU:  python -m nanoseek.scripts.pre_train
    Multi-GPU:   torchrun --nproc_per_node=8 -m nanoseek.scripts.pre_train

Adapts nanochat's base_train.py for NanoSeek's MoE + MLA + MTP architecture.
Key differences from nanochat:
    1. Model returns dict, not scalar loss
    2. muP 1/width scaling (not 1/√width)
    3. Cosine decay (not linear warmdown)
    4. Gradient clipping (MoE gradient variance)
    5. EMA tracking for all evaluation (RULE 1)
    6. MoE load-balance bias update after each step
    7. Batch size warmup (1/5 → 1× over first 10%)
    8. Phase transition (4K→8K at 80% of training)
"""
```

**Why document the 8 differences at the top?** Six months from now, when you're debugging a training run, you'll see this header and immediately know *every way* this script diverges from nanochat's well-tested baseline. Each numbered item is a code section below.

```python
import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
```

**What does `expandable_segments:True` do?**

```
Without it:
  PyTorch CUDA allocator reserves fixed-size memory blocks.
  MoE has VARIABLE memory usage per step (different experts activated
  → different intermediate tensor sizes). Fixed blocks fragment:

  GPU Memory: [████|░░░░|████|░░░░|████|░░░░|████]
               used  frag  used  frag  used  frag  used
               
  Eventually: OOM even though total free memory > needed size
  because no single CONTIGUOUS block is large enough.

With expandable_segments:True:
  Blocks can GROW in place, reducing fragmentation:
  
  GPU Memory: [████████████████████|░░░░░░░░░░░░░░░]
               used (one big block)  free (contiguous)
  
  Critical for MoE where expert activation patterns change every step.
```

```python
import gc
import json
import time
import math
import argparse
from dataclasses import asdict
from contextlib import contextmanager

import wandb
import torch
import torch.nn as nn
import torch.distributed as dist
from torch.nn.utils import clip_grad_norm_
```

**Why import `clip_grad_norm_` explicitly?** Nanochat doesn't clip. We clip at 1.0 because MoE gradient variance is 8× higher (each expert sees only κ=12.5% of tokens). This is our **Difference #4**.

```python
from nanoseek.nanoseek.config import (
    NanoSeekConfig,
    get_nanoseek_config,
    get_nanoseek_500m_config,
    get_nanoseek_anchor_config,
    get_training_phases,
    apply_phase_config,
)
from nanoseek.nanoseek.model import NanoSeekModel
```

**Why import all 3 config factories?** The same `pre-train.py` must handle all 3 scales in the muP transfer path. A CLI argument selects which config to use. The muP scaling code computes LR corrections relative to the anchor config.

```python
# Reuse nanochat infrastructure directly (no need to rewrite)
from nanochat.optim import MuonAdamW, DistMuonAdamW
from nanochat.common import (
    compute_init, compute_cleanup, print0,
    DummyWandb, autodetect_device_type, get_peak_flops,
    COMPUTE_DTYPE, is_ddp_initialized,
)
from nanochat.tokenizer import get_tokenizer, get_token_bytes
from nanochat.dataloader import (
    tokenizing_distributed_data_loader_bos_bestfit,
    tokenizing_distributed_data_loader_with_state_bos_bestfit,
)
from nanochat.loss_eval import evaluate_bpb
from nanochat.checkpoint_manager import save_checkpoint, load_checkpoint
```

### 🖼️ Visual — What We Reuse vs What We Write

```
FROM NANOCHAT (battle-tested, zero changes):
┌────────────────────────────────────────────────┐
│  optim.py        MuonAdamW / DistMuonAdamW     │ ← optimizer kernels
│  common.py       DDP init, print0, device      │ ← distributed setup
│  tokenizer.py    BPE tokenizer                  │ ← tokenization
│  dataloader.py   distributed data loading       │ ← data pipeline
│  loss_eval.py    BPB evaluation                 │ ← validation metric
│  checkpoint.py   save/load checkpoints          │ ← persistence
└────────────────────────────────────────────────┘

WE WRITE (MoE-specific, doesn't exist in nanochat):
┌────────────────────────────────────────────────┐
│  setup_optimizer()    muP param groups for MoE  │ ← Difference #4,5
│  get_lr_multiplier()  cosine decay schedule     │ ← Difference #3
│  batch_warmup()       1/5→1× ramp              │ ← Difference #7
│  training loop        dict loss + bias update   │ ← Difference #1,6
│  EMA tracking         CPU-side weight average   │ ← Difference #5
│  phase transition     4K→8K context switch      │ ← Difference #8
└────────────────────────────────────────────────┘
```

---

## Section 2: CLI Arguments

```python
parser = argparse.ArgumentParser(description="NanoSeek Stage 1 Pre-Training")

# Run configuration
parser.add_argument("--run", type=str, default="dummy",
                    help="wandb run name ('dummy' disables logging)")
parser.add_argument("--device-type", type=str, default="",
                    help="cuda|cpu|mps (empty = autodetect)")
```

These are identical to nanochat — same DDP/wandb infrastructure.

```python
# Model scale selection (muP transfer path)
parser.add_argument("--scale", type=str, default="1b",
                    choices=["anchor", "500m", "1b"],
                    help="which config to use: anchor(480h), 500m(1280h), 1b(2048h)")
```

**This is NEW.** Nanochat computes architecture from `--depth` and `--aspect-ratio`. NanoSeek uses pre-defined config factories because the MoE architecture has too many coupled parameters (E, top_k, moe_inter, MLA dimensions) for a simple formula. The `--scale` flag selects from our 3 muP-validated configs.

```
--scale anchor  →  get_nanoseek_anchor_config()   →  480h,  ~55M active
--scale 500m    →  get_nanoseek_500m_config()      →  1280h, ~441M active
--scale 1b      →  get_nanoseek_config()           →  2048h, ~1.08B active
```

```python
# Training horizon
parser.add_argument("--num-iterations", type=int, default=-1,
                    help="explicit step count (-1 = compute from config.total_tokens)")
parser.add_argument("--device-batch-size", type=int, default=16,
                    help="per-device micro-batch size (reduce if OOM)")
```

**Why default `device-batch-size=16` not 32?** MoE models use more memory per token than dense models. At 1B active, each forward pass activates 8 experts × 3 matrices + MLA decompression. Memory per token is ~1.5× a dense model of the same active params.

```python
# muP anchor reference (DO NOT CHANGE unless retuning anchor)
parser.add_argument("--mup-ref-width", type=int, default=480,
                    help="anchor hidden_size for muP scaling (must match anchor config)")
parser.add_argument("--mup-ref-batch-tokens", type=int, default=262144,
                    help="anchor batch size in tokens (64 × 4096)")

# muP base learning rates (tuned at anchor scale)
parser.add_argument("--matrix-lr", type=float, default=0.02,
                    help="base Muon LR for hidden weights (tuned at anchor)")
parser.add_argument("--embedding-lr", type=float, default=0.3,
                    help="base AdamW LR for embeddings (tuned at anchor)")
parser.add_argument("--unembedding-lr", type=float, default=0.008,
                    help="base AdamW LR for lm_head (tuned at anchor)")
parser.add_argument("--router-lr", type=float, default=3e-4,
                    help="AdamW LR for router weights (CONSTANT across scales)")
parser.add_argument("--norm-lr", type=float, default=3e-4,
                    help="AdamW LR for norm parameters (CONSTANT across scales)")
```

### 🖼️ Visual — Which LRs Scale, Which Don't

```
                        muP Scaling Rules
                        
Parameter Group    √(B/B_ref)    × (w_ref/w)    Net Effect at 1B
─────────────────────────────────────────────────────────────────
embedding          ✅ YES         ❌ NO           × 1.414
lm_head            ✅ YES         ❌ NO           × 1.414
Muon (hidden)      ✅ YES         ✅ YES          × 0.331
router             ❌ NO          ❌ NO           × 1.000
norms              ❌ NO          ❌ NO           × 1.000
                   ^^^^^^^^       ^^^^^^^^
                   Complete(d)P   Tensor Programs V
                   (batch size)   (width scaling)
```

```python
# Weight decay
parser.add_argument("--weight-decay", type=float, default=0.1,
                    help="base weight decay for Muon groups")

# Evaluation
parser.add_argument("--eval-every", type=int, default=250,
                    help="evaluate ema_val_bpb every N steps (-1 = disable)")
parser.add_argument("--eval-tokens", type=int, default=10_000_000,
                    help="number of tokens for validation evaluation")
parser.add_argument("--save-every", type=int, default=1000,
                    help="save checkpoint every N steps (-1 = only at end)")

# EMA configuration
parser.add_argument("--ema-decay", type=float, default=0.9999,
                    help="EMA decay rate (Polyak averaging)")
parser.add_argument("--ema-every", type=int, default=10,
                    help="update EMA every N steps")

# Resume
parser.add_argument("--resume-from-step", type=int, default=-1,
                    help="resume from checkpoint at this step")

args = parser.parse_args()
```

---

## Section 3: Compute Initialization

```python
# ─── Distributed setup ───
device_type = autodetect_device_type() if args.device_type == "" else args.device_type
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init(device_type)
master_process = ddp_rank == 0
synchronize = torch.cuda.synchronize if device_type == "cuda" else lambda: None
```

This is identical to nanochat. `compute_init()` handles:

```
Single GPU:  ddp=False, rank=0, world_size=1
torchrun:    ddp=True,  rank=0..7, world_size=8
             Sets up NCCL process group for all-reduce/scatter/gather
```

```python
# ─── GPU info ───
if device_type == "cuda":
    gpu_name = torch.cuda.get_device_name(0)
    gpu_peak_flops = get_peak_flops(gpu_name)
    print0(f"GPU: {gpu_name} | Peak BF16 FLOPS: {gpu_peak_flops:.2e}")
else:
    gpu_peak_flops = float('inf')

# ─── W&B ───
use_dummy_wandb = args.run == "dummy" or not master_process
wandb_run = DummyWandb() if use_dummy_wandb else wandb.init(
    project="nanoseek", name=args.run, config=vars(args)
)
```

---

## Section 4: Model Initialization

This is where things diverge significantly from nanochat.

```python
# ─── Select config from muP transfer path ───
config_map = {
    "anchor": get_nanoseek_anchor_config,
    "500m":   get_nanoseek_500m_config,
    "1b":     get_nanoseek_config,
}
config = config_map[args.scale]()
```

**Nanochat computes config from depth × aspect_ratio.** We can't do that because MoE has 20+ coupled parameters. The 3 factory functions guarantee muP-consistent ratios.

```python
print0(f"Model scale: {args.scale}")
print0(f"  hidden_size:   {config.hidden_size}")
print0(f"  num_layers:    {config.num_layers}")
print0(f"  n_experts:     {config.moe.n_routed_experts}")
print0(f"  top_k:         {config.moe.num_experts_per_tok}")
print0(f"  moe_inter:     {config.moe.moe_intermediate_size}")
print0(f"  vocab_size:    {config.vocab_size}")
```

Now, model initialization. Nanochat uses the `meta device` pattern:

```
Step 1: Build on meta device (shapes only, no data — 0 bytes GPU memory)
Step 2: to_empty(device=cuda) (allocate memory, but garbage data)
Step 3: init_weights() (fill with proper initializations)
```

**Why this 3-step dance?** Because NanoSeekModel has 4.75B total parameters. If we create it directly on GPU, we'd need 4.75B × 2 bytes (bf16) = 9.5 GB just for parameters, PLUS 9.5 GB for the initialization intermediates (randn creates temporaries). The meta device avoids the double memory spike.

```python
# ─── Build model ───
print0("Building model on meta device...")
with torch.device("meta"):
    model = NanoSeekModel(config)

# Move to device and initialize
model.to_empty(device=device)
model.init_weights()
```

```
Memory timeline during init:

                  meta device    to_empty()     init_weights()
GPU Memory:       0 GB           9.5 GB         9.5 GB (in place)
                  ↑ no data      ↑ garbage      ↑ proper values
                  just shapes    allocated       N(0, 0.02)
```

```python
# ─── Parameter counts ───
param_counts = model.num_parameters()
n_active = param_counts['active']
n_total = param_counts['total']
print0(f"Parameters: {n_active:,} active / {n_total:,} total "
       f"(expansion: {n_total/n_active:.1f}×)")
```

**Critical for muP**: all subsequent scaling uses `config.hidden_size`, NOT parameter counts. The parameter count tells us Chinchilla-optimal tokens. The hidden_size tells us muP LR scaling. These are independent axes.

---

## Section 5: FLOPs Estimation

Nanochat has `model.estimate_flops()`. NanoSeekModel doesn't — we compute it directly from first principles.

```python
# ─── FLOPs per token (for MFU calculation) ───
# Law 3 from RESEARCH_ENGINEER.md: FLOPs = 6 × N_active × D
# Per token: flops_per_token = 6 × N_active
# The "6" comes from: 2 (multiply-add) × 3 (forward + backward = 3×forward)
#
#   Forward pass:  each weight does 1 multiply + 1 add = 2 FLOPs per param per token
#   Backward pass: 2× forward (grad w.r.t. activations + grad w.r.t. weights)
#   Total:         2 + 4 = 6 FLOPs per param per token
#
num_flops_per_token = 6 * n_active
print0(f"FLOPs per token: {num_flops_per_token:.2e}")
print0(f"Total training FLOPs: {num_flops_per_token * config.total_tokens:.2e}")
```

### 🖼️ Visual — Where the "6" Comes From

```
For a single matrix multiply Y = X @ W,  where X:[B,d_in], W:[d_in, d_out]

Forward:   B × d_in × d_out multiplications
         + B × d_in × d_out additions          = 2 × B × d_in × d_out FLOPs

Backward (∂L/∂X):  same shape multiply         = 2 × B × d_in × d_out FLOPs

Backward (∂L/∂W):  same shape multiply         = 2 × B × d_in × d_out FLOPs

Total = 6 × B × d_in × d_out = 6 × B × (params in W)

Sum over all weight matrices → FLOPs = 6 × N_active × B
Per token (B=1):  FLOPs = 6 × N_active

CRITICAL: Use N_ACTIVE, not N_total!
  N_total = 4.75B (all expert weights)
  N_active = 1.08B (only 8 of 64 experts fire per token)
  
  Using N_total would overcount FLOPs by 4.4×
  This is Bug #6 from CLAUDE.md
```

---

## Section 6: muP Scaling — The Heart of HP Transfer

This is the most important section. Every line has mathematical justification.

```python
# ═══════════════════════════════════════════════════════════════════
# muP Hyperparameter Transfer (Tensor Programs V + Complete(d)P)
# ═══════════════════════════════════════════════════════════════════
#
# We have two scaling factors that compose multiplicatively:
#
# Factor 1: √(B/B_ref)  — from Complete(d)P (batch size scaling)
#   Larger batch → cleaner gradient → can take bigger step
#   η ∝ √B because gradient noise ∝ 1/√B
#
# Factor 2: w_ref/w     — from Tensor Programs V (width scaling)
#   Wider network → each weight contributes less to activation update
#   η ∝ 1/width to keep ||Δh|| = Θ(1) across widths
#
# Combined for hidden weights:  η = η_ref × √(B/B_ref) × (w_ref/w)
# For input/output weights:     η = η_ref × √(B/B_ref)
# For scale-independent params: η = η_ref (constant)
# ═══════════════════════════════════════════════════════════════════

w = config.hidden_size          # current model width
w_ref = args.mup_ref_width      # anchor width (480)

# Batch size in tokens
total_batch_tokens = config.global_batch_size * config.sequence_length
B_ref = args.mup_ref_batch_tokens  # anchor batch: 64 × 4096 = 262144

# Factor 1: √(B/B_ref) — Complete(d)P batch scaling
batch_lr_scale = math.sqrt(total_batch_tokens / B_ref)

# Factor 2: w_ref/w — Tensor Programs V width scaling
width_lr_scale = w_ref / w

print0(f"muP scaling factors:")
print0(f"  w_ref={w_ref}, w={w}")
print0(f"  B_ref={B_ref:,}, B={total_batch_tokens:,}")
print0(f"  √(B/B_ref) = {batch_lr_scale:.4f}")
print0(f"  w_ref/w    = {width_lr_scale:.4f}")
print0(f"  combined (hidden weights) = {batch_lr_scale * width_lr_scale:.4f}")
```

**Example output for 1B scale:**

```
muP scaling factors:
  w_ref=480, w=2048
  B_ref=262,144, B=524,288
  √(B/B_ref) = 1.4142
  w_ref/w    = 0.2344
  combined (hidden weights) = 0.3314
```

This means hidden weight LRs at 1B are **0.33× the anchor LR**. Without the width correction, they'd be **1.41× the anchor LR** — 4.27× too high!

---

## Section 7: Weight Decay Scaling (T_epoch Framework)

```python
# ═══════════════════════════════════════════════════════════════════
# Weight Decay Scaling — T_epoch framework (arXiv:2405.13698)
# ═══════════════════════════════════════════════════════════════════
#
# Central idea: T_epoch = B / (η × λ × D) should be constant.
# 
# We already scaled η by √(B/B_ref). To keep T_epoch constant:
#   λ = λ_ref × √(B/B_ref) × (D_ref / D)
#
# Where D is total training tokens.
# 
# Derivation:
#   T_epoch = B / (η × λ × D)
#   At reference: T_ref = B_ref / (η_ref × λ_ref × D_ref)
#   At target:    T_tgt = B / (η_ref × √(B/B_ref) × λ × D)
#   
#   Set T_tgt = T_ref:
#   B / (η_ref × √(B/B_ref) × λ × D) = B_ref / (η_ref × λ_ref × D_ref)
#   
#   Solve for λ:
#   λ = λ_ref × (B × D_ref) / (B_ref × D) × (1/√(B/B_ref))
#     = λ_ref × √(B/B_ref) × (D_ref/D)
#   ═══════════════════════════════════════════════════════════════════

D_ref = get_nanoseek_anchor_config().total_tokens  # anchor training tokens
D = config.total_tokens                              # current training tokens

weight_decay_scaled = args.weight_decay * batch_lr_scale * (D_ref / D)
print0(f"Weight decay: {args.weight_decay} → {weight_decay_scaled:.6f} "
       f"(T_epoch scaling: √B={batch_lr_scale:.4f} × D_ref/D={D_ref/D:.4f})")
```

**Example for 1B (D=22B, D_ref=1.1B):**

```
WD = 0.1 × 1.414 × (1.1B / 22B)
   = 0.1 × 1.414 × 0.05
   = 0.00707

The 1B model trains 20× longer than anchor → WD drops 20×
Combined with √B factor → net 14× reduction
```

### 🖼️ Visual — T_epoch Intuition

```
T_epoch = how many times each weight "sees" each training example
        = B / (η × λ × D)
        
If you train LONGER (D ↑):
  Each weight gets more gradient updates
  WD has more opportunities to shrink weights
  Without reducing λ: weights get over-regularized → underfitting

If you use BIGGER batches (B ↑):
  Each update is more accurate (less noise)
  You can use bigger η (√B scaling)
  But bigger η × λ = more WD per step
  Need to increase λ proportionally to maintain T_epoch

Net: λ ∝ √(B/B_ref) × (D_ref/D)
```

---

## Section 8: Optimizer Construction

This is where nanochat's `model.setup_optimizer()` gets replaced with our MoE-aware construction.

```python
# ═══════════════════════════════════════════════════════════════════
# Optimizer Construction — MoE-aware parameter groups
# ═══════════════════════════════════════════════════════════════════
#
# NanoSeekModel has NO setup_optimizer() method.
# We build param groups here because MoE requires careful classification:
#
#   Muon groups:  2D weight matrices EXCEPT embed, lm_head, gate.weight
#                 Grouped by shape for efficient stacking in optimizer
#                 LR: base × √(B/B_ref) × (w_ref/w)
#
#   AdamW groups:
#     embed:      LR: base × √(B/B_ref)         (input weight — no 1/w)
#     lm_head:    LR: base × √(B/B_ref)         (output weight — no 1/w)
#     router:     LR: constant                   (μP-MoE output weight)
#     norms:      LR: constant, no WD            (1D parameters)
# ═══════════════════════════════════════════════════════════════════

def setup_optimizer(model, config, args, batch_lr_scale, width_lr_scale,
                    weight_decay_scaled, ddp):
    """Build MoE-aware MuonAdamW optimizer with muP scaling.
    
    Parameter classification visual:
    
    NanoSeekModel
    ├── embed_tokens.weight          → AdamW (embedding)
    ├── layers[0..15]
    │   ├── self_attn
    │   │   ├── wq_a.weight          → Muon (2D hidden weight)
    │   │   ├── wq_b.weight          → Muon
    │   │   ├── wkv_a.weight         → Muon
    │   │   ├── wkv_b.weight         → Muon
    │   │   └── wo.weight            → Muon
    │   ├── input_layernorm.weight   → AdamW (norm, 1D)
    │   ├── post_attention_layernorm → AdamW (norm, 1D)
    │   ├── ffn (Dense, layers 0-1)
    │   │   ├── gate_proj.weight     → Muon (2D hidden weight)
    │   │   ├── up_proj.weight       → Muon
    │   │   └── down_proj.weight     → Muon
    │   └── ffn (MoE, layers 2-15)
    │       ├── gate.weight          → AdamW (ROUTER — constant LR)
    │       ├── shared_experts
    │       │   ├── gate_proj.weight → Muon (2D hidden weight)
    │       │   ├── up_proj.weight   → Muon
    │       │   └── down_proj.weight → Muon
    │       └── experts[0..63]
    │           ├── gate_proj.weight → Muon (2D hidden weight)
    │           ├── up_proj.weight   → Muon
    │           └── down_proj.weight → Muon
    ├── norm.weight                  → AdamW (norm, 1D)
    ├── lm_head.weight               → AdamW (lm_head)
    └── mtp (if exists)
        ├── proj.weight              → Muon (2D hidden weight)
        └── ...                      → Muon
    """
    
    # Classify every parameter
    embedding_params = []
    lm_head_params = []
    router_params = []
    norm_params = []
    muon_shapes = {}  # shape → [params] for Muon stacking
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
            
        # ─── Rule 1: Embedding (input weight) ───
        if 'embed_tokens' in name:
            embedding_params.append(param)
            
        # ─── Rule 2: LM head (output weight) ───
        elif 'lm_head' in name:
            lm_head_params.append(param)
            
        # ─── Rule 3: Router weights (muP output weight — CONSTANT LR) ───
        elif 'gate.weight' in name and 'gate_proj' not in name:
            # gate.weight = router, gate_proj.weight = SwiGLU first linear
            # Must distinguish: "layers.2.ffn.gate.weight" (router)
            #               vs  "layers.2.ffn.experts.0.gate_proj.weight" (SwiGLU)
            router_params.append(param)
            
        # ─── Rule 4: Norm parameters (1D, no weight decay) ───
        elif param.ndim == 1:
            # RMSNorm weights, biases, any 1D scalar
            norm_params.append(param)
            
        # ─── Rule 5: Everything else 2D → Muon ───
        elif param.ndim == 2:
            shape = param.shape
            muon_shapes.setdefault(shape, []).append(param)
            
        else:
            # 3D+ params (shouldn't exist in NanoSeek)
            raise ValueError(f"Unexpected param dim {param.ndim}: {name} {param.shape}")
```

### ⚠️ Critical Pitfall: `gate.weight` vs `gate_proj.weight`

```
In NanoSeek's MoE layer:
  
  ffn (MoEBlock)
  ├── gate           ← Router nn.Linear(hidden, n_experts)
  │   └── .weight    ← This is gate.weight → ROUTER (AdamW, constant LR)
  │
  ├── experts[0..63]
  │   └── Expert (SwiGLU)
  │       ├── gate_proj    ← First SwiGLU linear
  │       │   └── .weight  ← This is gate_proj.weight → MUON (hidden weight)
  │       ├── up_proj
  │       │   └── .weight  ← up_proj.weight → MUON
  │       └── down_proj
  │           └── .weight  ← down_proj.weight → MUON
  │
  └── shared_experts[0..1]
      └── (same structure as above)

The name "gate" appears in BOTH:
  "layers.2.ffn.gate.weight"                → ROUTER
  "layers.2.ffn.experts.0.gate_proj.weight" → SwiGLU

Our check: 'gate.weight' in name AND 'gate_proj' NOT in name
This is why we wrote: 
  elif 'gate.weight' in name and 'gate_proj' not in name
```

Now build the param groups:

```python
    # ─── Compute scaled LRs ───
    # Hidden weight LR: base × √(B/B_ref) × (w_ref/w)
    hidden_lr_scale = batch_lr_scale * width_lr_scale
    
    # Input/output weight LR: base × √(B/B_ref) only
    boundary_lr_scale = batch_lr_scale
    
    # Build param_groups list
    param_groups = []
    
    # AdamW groups
    if embedding_params:
        param_groups.append(dict(
            kind='adamw',
            params=embedding_params,
            lr=args.embedding_lr * boundary_lr_scale,
            betas=(0.9, 0.95),
            eps=1e-10,
            weight_decay=0.001,
        ))
    
    if lm_head_params:
        param_groups.append(dict(
            kind='adamw',
            params=lm_head_params,
            lr=args.unembedding_lr * boundary_lr_scale,
            betas=(0.9, 0.95),
            eps=1e-10,
            weight_decay=0.01,
        ))
    
    if router_params:
        param_groups.append(dict(
            kind='adamw',
            params=router_params,
            lr=args.router_lr,  # CONSTANT — no scaling
            betas=(0.9, 0.95),
            eps=1e-10,
            weight_decay=0.0,   # no WD for router
        ))
    
    if norm_params:
        param_groups.append(dict(
            kind='adamw',
            params=norm_params,
            lr=args.norm_lr,    # CONSTANT — no scaling
            betas=(0.9, 0.95),
            eps=1e-10,
            weight_decay=0.0,   # no WD for norms
        ))
    
    # Muon groups (one per unique shape, for stacking)
    for shape in sorted(muon_shapes.keys()):
        group_params = muon_shapes[shape]
        param_groups.append(dict(
            kind='muon',
            params=group_params,
            lr=args.matrix_lr * hidden_lr_scale,
            momentum=0.95,
            ns_steps=5,
            beta2=0.9,
            weight_decay=weight_decay_scaled,
        ))
```

### 🖼️ Visual — Parameter Group Shapes (1B config)

```
Muon groups (one group per unique shape):
Shape [2048, 440]     — wq_a projections          × 16 layers = 16 params
Shape [440, 2048]     — some MLA decompression     × varies
Shape [2048, 143]     — wkv_a projections          × 16 layers
Shape [143, 2048]     — some MLA decompression     × varies
Shape [2048, 768]     — expert down_proj + shared  × (14×64 + 14×2 + 2) = 952 params!
Shape [768, 2048]     — expert gate_proj/up_proj   × (14×64×2 + 14×2×2 + 2×2) = 1848 params!
Shape [2048, 5243]    — dense FFN gate/up          × 2 layers × 2 = 4 params
Shape [5243, 2048]    — dense FFN down             × 2 layers = 2 params
Shape [2048, 2048]    — wo projections             × 16 layers = 16 params
...

The [768, 2048] Muon group has 1848 parameters stacked!
The optimizer calls muon_step_fused on a single [1848, 768, 2048] tensor.
This is why grouping by shape matters — MASSIVE batched matrix operations.
```

```python
    # Create optimizer
    Factory = DistMuonAdamW if ddp else MuonAdamW
    optimizer = Factory(param_groups)
    
    # Store initial_lr for scheduler
    for group in optimizer.param_groups:
        group["initial_lr"] = group["lr"]
    
    # Logging
    n_muon = sum(len(g['params']) for g in param_groups if g['kind'] == 'muon')
    n_adamw = sum(len(g['params']) for g in param_groups if g['kind'] == 'adamw')
    print0(f"Optimizer: {Factory.__name__}")
    print0(f"  Muon params:  {n_muon} (in {sum(1 for g in param_groups if g['kind']=='muon')} shape groups)")
    print0(f"  AdamW params: {n_adamw} (embed={len(embedding_params)}, "
           f"lm_head={len(lm_head_params)}, router={len(router_params)}, "
           f"norm={len(norm_params)})")
    print0(f"  Hidden weight LR:  {args.matrix_lr * hidden_lr_scale:.6f} "
           f"(base={args.matrix_lr} × {hidden_lr_scale:.4f})")
    print0(f"  Embedding LR:      {args.embedding_lr * boundary_lr_scale:.6f}")
    print0(f"  LM head LR:        {args.unembedding_lr * boundary_lr_scale:.6f}")
    print0(f"  Router LR:         {args.router_lr} (constant)")
    print0(f"  Weight decay:      {weight_decay_scaled:.6f}")
    
    return optimizer
```

**Verification: no parameter left behind**

```python
    # Sanity check: every parameter accounted for
    n_total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n_grouped = sum(
        p.numel()
        for g in param_groups
        for p in g['params']
    )
    assert n_total_params == n_grouped, \
        f"Parameter mismatch: model has {n_total_params} but optimizer has {n_grouped}"
```

---

## Section 9: LR Schedule (Warmup → Constant → Cosine Decay)

```python
# ═══════════════════════════════════════════════════════════════════
# LR Schedule — warmup → constant → cosine decay
# ═══════════════════════════════════════════════════════════════════
#
# Nanochat: warmup → constant → LINEAR warmdown
# NanoSeek: warmup → constant → COSINE decay
#
# Why cosine instead of linear?
#   Linear:  LR drops at constant rate — sudden change at boundary
#   Cosine:  LR drops slowly, then faster — smooth S-curve
#   
#   For MoE: expert routing is sensitive to LR changes.
#   Sudden drops can freeze expert roles prematurely.
#   Cosine gives a gradual transition that lets experts adapt.
#
#           LR
#   1.0 ──┐
#          │  warmup   constant          cosine decay
#          │  /     ┌────────────┐
#          │ /      │            │╲
#          │/       │            │  ╲
#          │        │            │    ╲___________
#   0.1 ──│────────│────────────│────────────────── lr_min/lr
#          │        │            │
#          └────────┴────────────┴────────────────→ steps
#          0    warmup    warmup+          total
#               steps    constant          steps
#                        steps
# ═══════════════════════════════════════════════════════════════════

def get_lr_multiplier(step, warmup_steps, constant_steps, decay_steps, lr_min_ratio):
    """Returns multiplier in [lr_min_ratio, 1.0] for the current step.
    
    Args:
        step: current training step
        warmup_steps: number of linear warmup steps
        constant_steps: number of constant LR steps
        decay_steps: number of cosine decay steps
        lr_min_ratio: minimum LR as fraction of peak (e.g., 0.1 for lr_min/lr)
    """
    if step < warmup_steps:
        # Linear warmup: 0 → 1.0
        return (step + 1) / warmup_steps
    
    elif step < warmup_steps + constant_steps:
        # Constant phase: hold at 1.0
        return 1.0
    
    else:
        # Cosine decay: 1.0 → lr_min_ratio
        progress = (step - warmup_steps - constant_steps) / max(decay_steps, 1)
        progress = min(progress, 1.0)  # clamp to [0, 1]
        return lr_min_ratio + 0.5 * (1.0 - lr_min_ratio) * (1.0 + math.cos(math.pi * progress))
```

**Let's trace through the math with concrete numbers for 1B:**

```
config.warmup_steps = 1000
total_steps = 22B / (128 × 4096) = 22B / 524288 ≈ 41,961 steps

constant_steps = floor(0.70 × 41961) = 29,372
decay_start = 1000 + 29372 = 30,372
decay_end = floor(0.95 × 41961) = 39,862  
decay_steps = 39862 - 30372 = 9,490

lr_min_ratio = 3e-5 / 3e-4 = 0.1

Step 0:      (0+1)/1000 = 0.001                              LR ≈ 0
Step 500:    501/1000 = 0.501                                 LR = 50% peak
Step 1000:   1.0                                              LR = peak
Step 20000:  1.0                                              LR = peak
Step 30372:  1.0                                              LR = peak (decay starts)
Step 35117:  progress = 4745/9490 = 0.5                       
             0.1 + 0.5 × 0.9 × (1 + cos(π×0.5))
             = 0.1 + 0.45 × (1 + 0) = 0.55                   LR = 55% peak
Step 39862:  progress = 1.0
             0.1 + 0.5 × 0.9 × (1 + cos(π)) 
             = 0.1 + 0.45 × (1 - 1) = 0.1                    LR = 10% peak = lr_min
Step 41961:  still 0.1 (clamped)                              LR = lr_min
```

---

## Section 10: Muon Momentum Schedule

```python
# ─── Muon momentum warmup (from nanochat, works well) ───
def get_muon_momentum(step):
    """Warms momentum from 0.85 → 0.97 over first 400 steps.
    
    Why warm up momentum? At step 0, momentum buffer is empty.
    High momentum (0.97) with empty buffer = noisy.
    Start low (0.85), increase as buffer fills with signal.
    """
    frac = min(step / 400, 1.0)
    return (1 - frac) * 0.85 + frac * 0.97
```

---

## Section 11: Batch Size Warmup

```python
# ═══════════════════════════════════════════════════════════════════
# Batch Size Warmup — ramp from 1/5 → 1× of target
# ═══════════════════════════════════════════════════════════════════
#
# DeepSeek V3: ramps over first 3% of training (aggressive)
# NanoSeek:    ramps over first 10% of training (conservative)
#
# Mechanism: we DON'T change the data loader batch size.
# Instead, we reduce grad_accum_steps and scale loss accordingly.
# 
# Example for 1B:
#   target_batch = 128 × 4096 = 524,288 tokens
#   device_batch = 16 × 4096  = 65,536 tokens per GPU
#   world_batch  = 65,536 × 8 = 524,288 tokens per fwdbwd
#   target_accum = 524,288 / 524,288 = 1  (no accum needed for 8 GPUs)
#
# For anchor (4 GPUs):
#   target_batch = 64 × 4096  = 262,144 tokens
#   device_batch = 16 × 4096  = 65,536 tokens per GPU
#   world_batch  = 65,536 × 4 = 262,144 tokens per fwdbwd
#   target_accum = 262,144 / 262,144 = 1  (no accum needed)
#
# But if device_batch_size is smaller (e.g., 8 for OOM):
#   world_batch  = 8 × 4096 × 8 = 262,144
#   target_accum = 524,288 / 262,144 = 2  (need 2 accumulation steps)
#   warmup_accum = ceil(2 / 5) = 1        (start with 1 accum step)
# ═══════════════════════════════════════════════════════════════════

def get_batch_warmup_accum(step, target_accum, total_steps, warmup_fraction=0.10):
    """Get current grad accumulation steps during batch warmup.
    
    Ramps from ceil(target_accum/5) to target_accum over first
    warmup_fraction of training.
    
    Returns:
        current_accum: int, number of gradient accumulation steps this step
    """
    warmup_end = int(total_steps * warmup_fraction)
    
    if step >= warmup_end or target_accum <= 1:
        return target_accum
    
    min_accum = max(1, math.ceil(target_accum / 5))
    
    # Linear ramp from min_accum to target_accum
    progress = step / max(warmup_end, 1)
    current = min_accum + (target_accum - min_accum) * progress
    return max(min_accum, round(current))
```

---

## Section 12: EMA Tracker

```python
# ═══════════════════════════════════════════════════════════════════
# EMA Weight Tracker — CPU-side Polyak averaging
# ═══════════════════════════════════════════════════════════════════
#
# RULE 1: ALL evaluation uses EMA weights, never raw weights.
# RULE 3: Scaling law fit uses ema_val_bpb, not val_bpb.
#
# EMA formula: θ_ema = α × θ_ema + (1 - α) × θ_model
#   α = 0.9999 (decay rate)
#   Update every 10 steps (not every step — too expensive for 4.75B params)
#
# Why CPU-side?
#   EMA weights are 4.75B × 2 bytes = 9.5 GB in bf16.
#   If we keep them on GPU: doubles our parameter memory from 9.5→19 GB.
#   On CPU: free (RAM is cheap). Trade-off: slower eval (need to copy to GPU).
#   For eval every 250 steps, the copy cost is negligible.
#
# Why α=0.9999?
#   The "window" of EMA ≈ 1/(1-α) = 10,000 steps.
#   At 10 updates per step (ema_every=10): effective window = 1000 updates.
#   This smooths over ~1000 optimizer steps — filters SGD oscillation
#   around the loss minimum without being so long that it's stale.
#
#   Analogy: think of it as a low-pass filter.
#   Raw weights = noisy signal, oscillating around true optimum.
#   EMA weights = filtered signal, much closer to the true optimum.
#
#        Loss
#   ─────│──────────────────────────────────────
#        │   ╱╲   ╱╲   ╱╲
#        │  ╱  ╲ ╱  ╲ ╱  ╲   ← raw weights (oscillating)
#        │ ╱    ╳    ╳    ╲
#        │╱    ╱ ╲  ╱ ╲    ╲
#        │    ╱   ╲╱   ╲    ╲
#        │   ╱     │    ╲    ╲
#        │──╱──────│─────╲────╲─── ← EMA weights (smooth)
#        │ ╱       │      ╲    ╲
#        │╱        │       ╲    ╲
#   ─────┴─────────┴────────╲────╲── steps
#                            ╲    ╲
#                             ╲   optimum
# ═══════════════════════════════════════════════════════════════════

class EMATracker:
    """CPU-side Exponential Moving Average for model weights.
    
    Stores a shadow copy of all parameters on CPU.
    Updates every `update_every` optimizer steps.
    """
    
    def __init__(self, model, decay=0.9999, device='cpu'):
        self.decay = decay
        self.device = device
        # Deep copy all params to CPU
        self.shadow = {
            name: param.detach().clone().to(device)
            for name, param in model.named_parameters()
        }
    
    @torch.no_grad()
    def step(self, model):
        """Update EMA: shadow = decay × shadow + (1-decay) × model."""
        for name, param in model.named_parameters():
            shadow = self.shadow[name]
            # lerp_: shadow = shadow + (1-decay) × (param - shadow)
            #       = decay × shadow + (1-decay) × param
            shadow.lerp_(param.detach().to(self.device), 1 - self.decay)
    
    @contextmanager
    def apply(self, model):
        """Context manager: temporarily swap model weights with EMA weights.
        
        Usage:
            with ema_tracker.apply(model):
                val_bpb = evaluate_bpb(model, val_loader, ...)
        """
        # Save originals and swap in EMA
        originals = {}
        for name, param in model.named_parameters():
            originals[name] = param.detach().clone()
            param.data.copy_(self.shadow[name].to(param.device))
        
        try:
            yield
        finally:
            # Restore originals
            for name, param in model.named_parameters():
                param.data.copy_(originals[name])
    
    def state_dict(self):
        """For checkpoint saving."""
        return {k: v.clone() for k, v in self.shadow.items()}
    
    def load_state_dict(self, state_dict):
        """For checkpoint loading."""
        for k, v in state_dict.items():
            self.shadow[k] = v.to(self.device)
```

### 🖼️ Visual — EMA Memory Layout

```
                    GPU (CUDA)                    CPU (RAM)
              ┌───────────────────┐         ┌───────────────────┐
              │  model.parameters │         │  ema.shadow        │
              │                   │         │                    │
              │  embed: [65536,   │         │  embed: [65536,    │
              │         2048]     │         │         2048]      │
              │  = 256 MB         │         │  = 256 MB          │
              │                   │         │                    │
              │  experts:         │         │  experts:          │
              │  64×14×3×768×2048 │         │  64×14×3×768×2048  │
              │  = ~8 GB          │         │  = ~8 GB           │
              │                   │         │                    │
              │  total: ~9.5 GB   │         │  total: ~9.5 GB    │
              └───────┬───────────┘         └───────┬────────────┘
                      │                             │
                      │  every 10 steps:            │
                      │  shadow.lerp_(param, 0.0001)│
                      └─────────────────────────────┘
                      
              │  For eval (every 250 steps):
              │  copy shadow → GPU → eval → restore originals
              │  Cost: ~2 seconds (9.5 GB over PCIe)
              │  vs training step: ~200ms
              │  Overhead: 2s / (250 × 0.2s) = 4% — negligible
```

---

## Section 13: Training Loop Setup

```python
# ═══════════════════════════════════════════════════════════════════
# Training Loop Setup
# ═══════════════════════════════════════════════════════════════════

# ─── Tokenizer and data ───
tokenizer = get_tokenizer()
token_bytes = get_token_bytes(device=device)
print0(f"Vocab size: {tokenizer.get_vocab_size():,}")

# ─── Number of training iterations ───
total_batch_tokens = config.global_batch_size * config.sequence_length
if args.num_iterations > 0:
    num_iterations = args.num_iterations
else:
    num_iterations = config.total_tokens // total_batch_tokens
print0(f"Total iterations: {num_iterations:,}")
print0(f"Total tokens: {total_batch_tokens * num_iterations:,}")

# ─── LR schedule phase boundaries ───
warmup_steps = config.warmup_steps
total_after_warmup = num_iterations - warmup_steps
constant_steps = int(config.constant_phase_ratio * num_iterations)
decay_end_step = int(config.cosine_decay_end_ratio * num_iterations)
decay_steps = decay_end_step - warmup_steps - constant_steps
lr_min_ratio = config.lr_min / config.learning_rate  # 3e-5/3e-4 = 0.1

print0(f"LR schedule: warmup={warmup_steps}, constant={constant_steps}, "
       f"decay={decay_steps}, lr_min_ratio={lr_min_ratio:.2f}")
```

```python
# ─── Gradient accumulation ───
tokens_per_microbatch = args.device_batch_size * config.sequence_length
world_tokens_per_fwdbwd = tokens_per_microbatch * ddp_world_size
assert total_batch_tokens % world_tokens_per_fwdbwd == 0, \
    f"total_batch_tokens ({total_batch_tokens}) must be divisible by " \
    f"world_tokens_per_fwdbwd ({world_tokens_per_fwdbwd})"
target_grad_accum = total_batch_tokens // world_tokens_per_fwdbwd

print0(f"Gradient accumulation: {target_grad_accum} steps "
       f"({args.device_batch_size}×{config.sequence_length}×{ddp_world_size} "
       f"= {world_tokens_per_fwdbwd:,} tokens/fwdbwd)")
```

### 🖼️ Visual — Gradient Accumulation

```
For 1B on 8×H100:
  total_batch = 128 × 4096 = 524,288 tokens
  per_device  = 16 × 4096  = 65,536 tokens
  world       = 65,536 × 8 = 524,288 tokens
  accum       = 524,288 / 524,288 = 1  (no accumulation needed!)

For 1B on 4×H100 (less GPUs):
  world       = 65,536 × 4 = 262,144 tokens
  accum       = 524,288 / 262,144 = 2  (two micro-batches per step)
  
  Step N:
    micro-batch 1: forward → backward (accumulate grad)
    micro-batch 2: forward → backward (accumulate grad)
    clip → optimizer.step → zero_grad
    
For anchor on 4×A100 (smaller batch):
  total_batch = 64 × 4096  = 262,144 tokens
  per_device  = 16 × 4096  = 65,536 tokens  
  world       = 65,536 × 4 = 262,144 tokens
  accum       = 262,144 / 262,144 = 1

During batch warmup (first 10% of steps):
  accum starts at max(1, ceil(target/5))
  For target_accum=2: warmup_accum=1 → grows to 2
  For target_accum=1: no warmup (already at minimum)
```

```python
# ─── Build optimizer ───
optimizer = setup_optimizer(
    model, config, args,
    batch_lr_scale, width_lr_scale, weight_decay_scaled, ddp,
)

# ─── Compile model ───
orig_model = model  # keep reference for saving/eval
model = torch.compile(model, dynamic=False)

# ─── EMA tracker ───
ema_tracker = EMATracker(orig_model, decay=args.ema_decay)
print0(f"EMA tracker initialized (decay={args.ema_decay}, update every {args.ema_every} steps)")

# ─── Data loaders ───
train_loader = tokenizing_distributed_data_loader_with_state_bos_bestfit(
    tokenizer, args.device_batch_size, config.sequence_length,
    split="train", device=device, resume_state_dict=None,
)
build_val_loader = lambda: tokenizing_distributed_data_loader_bos_bestfit(
    tokenizer, args.device_batch_size, config.sequence_length,
    split="val", device=device,
)

# ─── Kick off first batch ───
x, y, dataloader_state_dict = next(train_loader)
```

---

## Section 14: The Training Loop

This is the core. Every line maps back to our data flow diagram.

```python
# ═══════════════════════════════════════════════════════════════════
# Training Loop
# ═══════════════════════════════════════════════════════════════════
#
# Data flow per step:
#
#   x, y ──→ model(x, labels=y, ...) ──→ outputs dict
#                                          │
#                    ┌─────────────────────┼───────────────────┐
#                    │                     │                   │
#                outputs['loss']    outputs['aux_loss']    _layer_aux_data
#                    │                     │                   │
#                    ▼                     │                   │
#              loss.backward()             │                   │
#              clip_grad_norm_()           │                   │
#              optimizer.step()            │                   │
#              zero_grad()                 │                   │
#                    │                     │                   │
#                    ▼                     ▼                   ▼
#           update_load_balance_bias()   wandb log         H_load stats
#           ema_tracker.step()
# ═══════════════════════════════════════════════════════════════════

step = 0
tokens_processed = 0
smooth_train_loss = 0.0
min_ema_val_bpb = float('inf')
total_training_time = 0.0

while True:
    last_step = step == num_iterations
    
    # ─────────────────────────────────────────────────────────────
    # EVALUATION (uses EMA weights per RULE 1)
    # ─────────────────────────────────────────────────────────────
    if args.eval_every > 0 and (last_step or step % args.eval_every == 0):
        model.eval()
        eval_steps = args.eval_tokens // (
            args.device_batch_size * config.sequence_length * ddp_world_size
        )
        val_loader = build_val_loader()
        
        # RULE 1: ALL eval uses EMA weights
        with ema_tracker.apply(orig_model):
            ema_val_bpb = evaluate_bpb(orig_model, val_loader, eval_steps, token_bytes)
        
        print0(f"Step {step:05d} | ema_val_bpb: {ema_val_bpb:.6f}")
        if ema_val_bpb < min_ema_val_bpb:
            min_ema_val_bpb = ema_val_bpb
        
        wandb_run.log({
            "step": step,
            "tokens_processed": tokens_processed,
            "val/ema_bpb": ema_val_bpb,
        })
        model.train()
```

**Why `ema_tracker.apply(orig_model)` uses `orig_model`?** The compiled model (`model = torch.compile(...)`) has optimized graphs that assume fixed parameter values. Swapping weights under the compiled model would trigger recompilation. Using `orig_model` (uncompiled) for evaluation avoids this.

```python
    # ─────────────────────────────────────────────────────────────
    # CHECKPOINT SAVING
    # ─────────────────────────────────────────────────────────────
    if last_step or (step > 0 and args.save_every > 0 and step % args.save_every == 0):
        checkpoint_dir = os.path.join("checkpoints", f"nanoseek_{args.scale}")
        save_checkpoint(
            checkpoint_dir,
            step,
            orig_model.state_dict(),
            optimizer.state_dict(),
            {
                "step": step,
                "tokens_processed": tokens_processed,
                "ema_val_bpb": ema_val_bpb if 'ema_val_bpb' in dir() else None,
                "config": asdict(config),
                "ema_state_dict": ema_tracker.state_dict(),
                "dataloader_state_dict": dataloader_state_dict,
                "loop_state": {
                    "min_ema_val_bpb": min_ema_val_bpb,
                    "smooth_train_loss": smooth_train_loss,
                    "total_training_time": total_training_time,
                },
            },
            rank=ddp_rank,
        )
    
    # ─── Termination ───
    if last_step:
        break
```

Now the actual training step:

```python
    # ═════════════════════════════════════════════════════════════
    # SINGLE TRAINING STEP
    # ═════════════════════════════════════════════════════════════
    
    # ─── Batch warmup: adjust grad accumulation ───
    current_accum = get_batch_warmup_accum(
        step, target_grad_accum, num_iterations
    )
    current_batch_tokens = world_tokens_per_fwdbwd * current_accum
    
    synchronize()
    t0 = time.time()
    
    # ─── Forward + Backward (with gradient accumulation) ───
    for micro_step in range(current_accum):
```

**This is Difference #1: model returns dict, not scalar.**

Nanochat:
```python
loss = model(x, y)           # returns scalar
```

NanoSeek:
```python
outputs = model(x, labels=y, # returns dict
    tokens_processed=tokens_processed,
    total_tokens=config.total_tokens)
loss = outputs['loss']        # extract scalar
```

```python
        outputs = model(
            x,
            labels=y,
            tokens_processed=tokens_processed,
            total_tokens=config.total_tokens,
        )
        loss = outputs['loss']
        train_loss = loss.detach()
        
        # Scale loss for gradient accumulation
        # Each .backward() ADDS to .grad → divide by accum count
        loss = loss / current_accum
        loss.backward()
        
        # Prefetch next batch while GPU does backward
        x, y, dataloader_state_dict = next(train_loader)
```

### 🖼️ Visual — Gradient Accumulation Math

```
Why divide loss by current_accum?

grad_accum=2 means 2 micro-batches per optimizer step.
PyTorch .backward() ADDS gradients (doesn't replace).

Without /accum:
  micro 1: grad += ∇L₁         (full gradient of batch 1)
  micro 2: grad += ∇L₂         (full gradient of batch 2)
  total:   grad = ∇L₁ + ∇L₂   (sum, not average!)
  → This is 2× the intended gradient magnitude
  → Equivalent to 2× learning rate → training diverges

With /accum:
  micro 1: grad += ∇L₁/2
  micro 2: grad += ∇L₂/2
  total:   grad = (∇L₁ + ∇L₂)/2 = ∇L_batch  (correct average)
  → Mathematically identical to processing the full batch at once
```

Now the optimizer step:

```python
    # ─── Gradient clipping (Difference #3) ───
    # MoE gradient variance is 8× higher than dense (κ=12.5%)
    # Without clipping: P(NaN within 1000 steps) ≈ 1 for bf16
    grad_norm = clip_grad_norm_(orig_model.parameters(), max_norm=1.0)
    
    # ─── Update learning rates ───
    lrm = get_lr_multiplier(step, warmup_steps, constant_steps,
                            decay_steps, lr_min_ratio)
    muon_momentum = get_muon_momentum(step)
    
    for group in optimizer.param_groups:
        group["lr"] = group["initial_lr"] * lrm
        if group['kind'] == 'muon':
            group["momentum"] = muon_momentum
```

**Note**: We DON'T have a weight decay schedule like nanochat's `get_weight_decay()` (cosine to 0). Our weight decay is already scaled via T_epoch at initialization. The LR schedule implicitly reduces the effective WD (since effective_WD ∝ lr × λ, and lr drops via cosine). We could add a separate WD schedule, but sweeps1.md doesn't specify one, and DeepSeek V3 uses constant WD.

```python
    # ─── Optimizer step ───
    optimizer.step()
    model.zero_grad(set_to_none=True)
```

Now the MoE-specific post-step updates:

```python
    # ─── MoE load-balance bias update (Difference #6) ───
    # This is the aux-loss-free balancing mechanism from DeepSeek V3.
    # After each gradient step, adjust expert biases to redistribute load.
    # Formula: b_i -= gamma × (load_i - mean) / mean
    # Gamma freezes at 0 after 95% of training (RULE 2).
    orig_model.update_load_balance_bias(tokens_processed, config.total_tokens)
    
    # ─── EMA update (Difference #5) ───
    if step % args.ema_every == 0:
        ema_tracker.step(orig_model)
    
    # ─── Update token counter ───
    tokens_processed += current_batch_tokens
```

### 🖼️ Visual — Load Balance Bias Update

```
After each step, for each MoE layer:

Expert loads from forward pass (cached in _layer_aux_data):
  load_counts = [1240, 980, 1100, 870, ...]  ← tokens per expert
  mean_load = mean(load_counts) = 1024

Bias update:
  b_i -= gamma × (load_i - mean) / mean
  
  Expert 0: 1240 tokens (overloaded)
    b_0 -= 0.001 × (1240 - 1024) / 1024 = -0.000211
    → bias decreases → router score decreases → fewer tokens next time
    
  Expert 3: 870 tokens (underloaded)
    b_3 -= 0.001 × (870 - 1024) / 1024 = +0.000150
    → bias increases → router score increases → more tokens next time

This is a CONTROL LOOP:
  
  Overloaded ──→ lower bias ──→ fewer tokens ──→ balanced
       ↑                                            │
       └────────────────────────────────────────────┘
  
After 95% of training: gamma=0 → biases freeze
  Why? Late-stage expert roles are established. Further
  rebalancing would disrupt learned specialization.
```

```python
    # ─── Timing ───
    train_loss_f = train_loss.item()
    synchronize()
    t1 = time.time()
    dt = t1 - t0
    
    if step > 10:
        total_training_time += dt
```

---

## Section 15: Logging

```python
    # ─── Logging ───
    ema_beta = 0.9
    smooth_train_loss = ema_beta * smooth_train_loss + (1 - ema_beta) * train_loss_f
    debiased_loss = smooth_train_loss / (1 - ema_beta ** (step + 1))
    
    tok_per_sec = int(current_batch_tokens / dt)
    flops_per_sec = num_flops_per_token * current_batch_tokens / dt
    mfu = 100 * flops_per_sec / (gpu_peak_flops * ddp_world_size)
    
    # H_load (expert balance entropy) — every 10 steps
    load_stats = orig_model.get_expert_load_stats()
    H_load = load_stats['entropy'].item()
    
    # Get current gamma for logging
    gamma = orig_model.get_gamma(tokens_processed, config.total_tokens)
    
    pct_done = 100 * step / num_iterations
    print0(
        f"step {step:05d}/{num_iterations} ({pct_done:.1f}%) | "
        f"loss: {debiased_loss:.4f} | H_load: {H_load:.2f} | "
        f"lr×: {lrm:.3f} | γ: {gamma:.4f} | "
        f"grad: {grad_norm:.2f} | "
        f"batch: {current_batch_tokens:,} | "
        f"mfu: {mfu:.1f}% | "
        f"tok/s: {tok_per_sec:,}"
    )
    
    if step % config.log_every_steps == 0:
        wandb_run.log({
            "step": step,
            "tokens_processed": tokens_processed,
            "train/loss": debiased_loss,
            "train/lr_multiplier": lrm,
            "train/grad_norm": grad_norm,
            "train/H_load": H_load,
            "train/gamma": gamma,
            "train/mfu": mfu,
            "train/tok_per_sec": tok_per_sec,
            "train/batch_tokens": current_batch_tokens,
            "train/aux_loss": outputs['aux_loss'].item(),
        })
    
    step += 1
```

### ⚠️ Critical: H_load Monitoring

```
H_load = entropy of expert load distribution

H_load = -Σᵢ pᵢ × log₂(pᵢ)   where pᵢ = load_i / Σ load_j

For 64 experts:
  Perfect balance:  H = log₂(64) = 6.0 bits (all experts equal)
  Complete collapse: H = 0 bits (one expert gets everything)
  Alert threshold:  H < 2.0 bits (fewer than 4 effective experts)

Timeline of a healthy run:
  Step 0:     H ≈ 6.0 (random routing, uniform)
  Step 1000:  H ≈ 5.5 (slight specialization, normal)
  Step 10000: H ≈ 4.5 (clear specialization, healthy)
  Step 40000: H ≈ 4.0 (mature specialization, good)

Timeline of COLLAPSED run:
  Step 0:     H ≈ 6.0 (random, looks fine)
  Step 500:   H ≈ 5.0 (slight drop, still looks fine)
  Step 1000:  H ≈ 3.5 (dropping... ⚠️ early warning)
  Step 2000:  H ≈ 1.5 (🚨 COLLAPSE — 3 experts do all work)
  Step 5000:  H ≈ 0.3 (dead — 90% of params wasted)
  
  Loss PLATEAU doesn't appear until step 3000-5000.
  H_load catches collapse 2000 steps BEFORE loss shows it.
  This is why H_load monitoring is a MUST-HAVE (Component H in sweeps1.md).
```

---

## Section 16: Cleanup

```python
# ─── End of training ───
print0(f"Training complete!")
print0(f"Total training time: {total_training_time/60:.1f} minutes")
print0(f"Min ema_val_bpb: {min_ema_val_bpb:.6f}")
print0(f"Peak memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB")

wandb_run.finish()
compute_cleanup()
```

---

## Full Code Summary — What We Built

```
Section  Lines  What                              Difference from nanochat
───────  ─────  ────────────────────────────────  ─────────────────────────
1-2      ~60    Imports + CLI args                --scale flag, muP params
3        ~15    Compute init                      Same as nanochat
4        ~20    Model init (meta device)          Uses NanoSeekConfig factories
5        ~10    FLOPs (6 × N_active)              Uses N_active not N_total
6        ~20    muP scaling factors               1/width + √B (NEW)
7        ~15    Weight decay (T_epoch)            Same formula as nanochat
8        ~100   Optimizer construction            MoE param groups (NEW)
9        ~20    LR schedule                       Cosine decay (not linear)
10       ~5     Muon momentum                     Same as nanochat
11       ~20    Batch warmup                      1/5→1× ramp (NEW)
12       ~60    EMA tracker class                 CPU-side EMA (NEW)
13       ~40    Training setup                    Connects all components
14       ~60    Training loop                     Dict loss, bias update (NEW)
15       ~40    Logging                           H_load, gamma, grad_norm
16       ~10    Cleanup                           Same as nanochat
TOTAL   ~495    lines                             ~300 lines are NanoSeek-specific
```

---

## ❓ Check Your Understanding

**Q1**: In the training loop, we call `orig_model.update_load_balance_bias()` AFTER `optimizer.step()`. Why is the ordering important? What would happen if we called it BEFORE the optimizer step?

**Q2**: The EMA tracker updates every 10 steps. At step 250 when we evaluate, the EMA has been updated 25 times. The effective smoothing window is `1/(1-0.9999) = 10,000` optimizer steps, but we've only done 250 steps. What does this mean for early evaluations? (Hint: think about EMA bias.)

**Q3**: During batch warmup, the effective batch size changes each step. But the LR schedule was computed assuming a FIXED batch size. Is this a problem? (Hint: the √B scaling in muP was computed at init time with `total_batch_tokens`.)

🇻🇳 **Tóm tắt**: Đã xây dựng hoàn chỉnh `pre-train.py` (~495 dòng) từ first principles, giải thích từng dòng code. 8 điểm khác biệt chính so với nanochat: (1) model trả dict không phải scalar, (2) loss combination bên trong model, (3) gradient clipping 1.0, (4) optimizer param groups cho MoE với muP 1/width, (5) EMA tracking CPU-side, (6) MoE bias update sau mỗi step, (7) batch warmup 1/5→1×, (8) cosine decay thay vì linear warmdown. Mỗi section có visual diagram và derivation toán học.